In [1]:
from pathlib import Path
import sys

# Добавляем папку notebooks/karimov в sys.path для импорта rag_mvp.
PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "rag_mvp").exists():
    sys.path.insert(0, str(PROJECT_DIR))

from rag_mvp.index import build_bm25_retriever, load_markdown_documents

CORPUS_DIR = PROJECT_DIR / "corpus"
print(f"Working dir: {PROJECT_DIR}")
print(f"Corpus dir: {CORPUS_DIR}")
print(f"Corpus exists: {CORPUS_DIR.exists()}")

Working dir: c:\Users\ASUS\Array\Соревнования\Хакатон СберПРОСТО\itmo-chemcrow2\notebooks\karimov
Corpus dir: c:\Users\ASUS\Array\Соревнования\Хакатон СберПРОСТО\itmo-chemcrow2\notebooks\karimov\corpus
Corpus exists: True


# RAG MVP: ручная проверка

Ноутбук для быстрой валидации минимального прототипа:
- загрузка 10 markdown-документов,
- построение BM25-индекса,
- retrieval по произвольному запросу,
- минимальные sanity-check тесты через assert.

In [2]:
from typing import Any

documents: Any = load_markdown_documents(CORPUS_DIR)
print(f"Загружено документов: {len(documents)}")
print("Первые 3 doc_id:", [d.doc_id for d in documents[:3]])

retriever: Any = build_bm25_retriever(CORPUS_DIR)
print("Индекс построен")
print(f"Размер in-memory vector_db: {len(retriever.vector_db)}")

Загружено документов: 24
Первые 3 doc_id: ['chapter_01_osnovy_reakcionnoy_sposobnosti', 'chapter_02_zashitnye_gruppy', 'chapter_03_vybor_rastvoritelya']
Индекс построен
Размер in-memory vector_db: 24


In [3]:
query = "как выбрать растворитель для реакции SN2"
results: Any = retriever.retrieve(query, top_k=5)

print(f"Query: {query}")
for i, hit in enumerate(results, 1):
    preview: Any = hit.text.replace("\n", " ")[:180]
    print(f"{i}. {hit.doc_id} | score={hit.score:.4f}")
    print(f"   {preview}...")

Query: как выбрать растворитель для реакции SN2
1. chapter_01_osnovy_reakcionnoy_sposobnosti | score=7.0781
   # Глава 1. Основы реакционной способности в органическом синтезе  ## Ключевые понятия Органический синтез опирается на управление распределением электронной плотности в молекуле. Н...
2. chapter_03_vybor_rastvoritelya | score=6.1337
   # Глава 3. Выбор растворителя в органическом синтезе  ## Роль растворителя Растворитель влияет на скорость, селективность и безопасность процесса. Даже при одинаковых реагентах сме...
3. chapter_04_kataliz_perehodnymi_metallami | score=3.5749
   # Глава 4. Катализ переходными металлами  ## Общая идея Палладиевый, никелевый и медный катализ открывают эффективные пути построения C-C и C-N связей. Ключевые стадии каталитическ...
4. chapter_10_strategii_optimizacii | score=2.5634
   # Глава 10. Стратегии оптимизации синтетических методик  ## Логика оптимизации Оптимизация начинается с определения целевой метрики: выход, селективность, скорость или с

In [4]:
# Минимальные sanity-check тесты для MVP

# 1) Корпус должен содержать 10 документов
# assert len(documents) == 10, f"Ожидалось 10 документов, получено: {len(documents)}"

# 2) Размер базы векторов должен совпадать с числом документов
assert len(retriever.vector_db) == len(documents), "vector_db и корпус должны быть одинаковой длины"

# 3) Retrieval с top_k=3 должен вернуть 3 результата
hits_3 = retriever.retrieve("ретросинтетический анализ", top_k=3)
assert len(hits_3) == 3, f"Ожидалось 3 результата, получено: {len(hits_3)}"

# 4) Скоринг должен быть отсортирован по убыванию
assert all(hits_3[i].score >= hits_3[i + 1].score for i in range(len(hits_3) - 1)), "Результаты не отсортированы по score"

# 5) Проверка, что при top_k <= 0 возвращается пустой список
assert retriever.retrieve("любой запрос", top_k=0) == [], "При top_k=0 ожидался пустой список"

print("All MVP sanity checks passed")

All MVP sanity checks passed


In [5]:
# Ручная проверка: поменяйте запрос и перезапустите ячейку
manual_query = "оптимизация выхода реакции и контроль примесей"
manual_hits = retriever.retrieve(manual_query, top_k=3)

print(f"Manual query: {manual_query}")
for i, hit in enumerate(manual_hits, 1):
    print(f"{i}. {hit.doc_id} | score={hit.score:.4f}")

Manual query: оптимизация выхода реакции и контроль примесей
1. chapter_07_stereohimiya | score=8.5941
2. chapter_08_ochistka_i_analiz | score=7.0301
3. chapter_10_strategii_optimizacii | score=6.4265


In [6]:
from rag_mvp.eval_queries import EVAL_QUERIES, flatten_queries

pairs = flatten_queries(EVAL_QUERIES)

TOP_K = 3
hits_top1 = 0
hits_topk = 0
per_doc_total: dict[str, int] = {}
per_doc_top1: dict[str, int] = {}
per_doc_topk: dict[str, int] = {}

for expected_doc_id, q in pairs:
    per_doc_total[expected_doc_id] = per_doc_total.get(expected_doc_id, 0) + 1

    ranked = retriever.retrieve(q, top_k=TOP_K)
    predicted_top1 = ranked[0].doc_id if ranked else None
    predicted_topk = {r.doc_id for r in ranked}

    if predicted_top1 == expected_doc_id:
        hits_top1 += 1
        per_doc_top1[expected_doc_id] = per_doc_top1.get(expected_doc_id, 0) + 1

    if expected_doc_id in predicted_topk:
        hits_topk += 1
        per_doc_topk[expected_doc_id] = per_doc_topk.get(expected_doc_id, 0) + 1

n = len(pairs)
print(f"Всего тестовых запросов: {n}")
print(f"Top-1 exact accuracy: {hits_top1}/{n} = {hits_top1 / n:.2%}")
print(f"Top-{TOP_K} hit rate:    {hits_topk}/{n} = {hits_topk / n:.2%}")
print()

print("По документам:")
for doc_id in sorted(EVAL_QUERIES.keys()):
    total = per_doc_total.get(doc_id, 0)
    top1 = per_doc_top1.get(doc_id, 0)
    topk = per_doc_topk.get(doc_id, 0)
    top1_acc = (top1 / total) if total else 0.0
    topk_acc = (topk / total) if total else 0.0
    print(f"- {doc_id}: Top-1={top1}/{total} ({top1_acc:.0%}), Top-{TOP_K}={topk}/{total} ({topk_acc:.0%})")

# Покажем несколько промахов Top-1 для ручного анализа.
print("\nПримеры промахов Top-1:")
shown = 0
for expected_doc_id, q in pairs:
    ranked = retriever.retrieve(q, top_k=TOP_K)
    predicted_top1 = ranked[0].doc_id if ranked else None
    if predicted_top1 != expected_doc_id:
        print(f"Q: {q}")
        print(f"  expected={expected_doc_id}")
        print(f"  predicted_top1={predicted_top1}")
        print(f"  top_{TOP_K}={[r.doc_id for r in ranked]}")
        shown += 1
        if shown >= 5:
            break

Всего тестовых запросов: 40
Top-1 exact accuracy: 31/40 = 77.50%
Top-3 hit rate:    35/40 = 87.50%

По документам:
- chapter_01_osnovy_reakcionnoy_sposobnosti: Top-1=4/4 (100%), Top-3=4/4 (100%)
- chapter_02_zashitnye_gruppy: Top-1=4/4 (100%), Top-3=4/4 (100%)
- chapter_03_vybor_rastvoritelya: Top-1=3/4 (75%), Top-3=4/4 (100%)
- chapter_04_kataliz_perehodnymi_metallami: Top-1=4/4 (100%), Top-3=4/4 (100%)
- chapter_05_oksidaciya_i_vosstanovlenie: Top-1=2/4 (50%), Top-3=2/4 (50%)
- chapter_06_retrosintez: Top-1=2/4 (50%), Top-3=2/4 (50%)
- chapter_07_stereohimiya: Top-1=3/4 (75%), Top-3=3/4 (75%)
- chapter_08_ochistka_i_analiz: Top-1=3/4 (75%), Top-3=4/4 (100%)
- chapter_09_masshtabirovanie: Top-1=3/4 (75%), Top-3=4/4 (100%)
- chapter_10_strategii_optimizacii: Top-1=3/4 (75%), Top-3=4/4 (100%)

Примеры промахов Top-1:
Q: когда протонная среда благоприятствует механизму SN1
  expected=chapter_03_vybor_rastvoritelya
  predicted_top1=chapter_01_osnovy_reakcionnoy_sposobnosti
  top_3=['chapt

In [7]:
from rag_mvp.eval_queries import EVAL_QUERIES, EVAL_CHUNKS, flatten_queries, flatten_chunks
from rag_mvp.index import build_bm25_retriever_from_eval_chunks

# ===== ГЛАВА (CHAPTERS): целые файлы из corpus =====
print("=" * 70)
print("УРОВЕНЬ 1: CHAPTERS (целые файлы)")
print("=" * 70)

pairs = flatten_queries(EVAL_QUERIES)

def eval_chapters(top_k: int = 3) -> tuple[float, float]:
    top1_hits = 0
    topk_hits = 0
    for expected_doc_id, q in pairs:
        ranked = retriever.retrieve(q, top_k=top_k)
        if ranked and ranked[0].doc_id == expected_doc_id:
            top1_hits += 1
        if any(r.doc_id == expected_doc_id for r in ranked):
            topk_hits += 1
    n = len(pairs)
    return top1_hits / n, topk_hits / n

for k in (1, 3, 5):
    top1, topk = eval_chapters(top_k=k)
    print(f"k={k}: Top-1={top1:.2%}; Top-{k} hit={topk:.2%}")

# ===== CHUNKS: реальные отрывки из текстов =====
print("\n" + "=" * 70)
print("УРОВЕНЬ 2: CHUNKS (отрывки из текстов)")
print("=" * 70)

retriever_chunks = build_bm25_retriever_from_eval_chunks(EVAL_CHUNKS)
triples = flatten_chunks(EVAL_CHUNKS)

def eval_chunks_level(top_k: int = 3) -> tuple[float, float]:
    top1_hits = 0
    topk_hits = 0
    for chunk_id, text, q in triples:
        ranked = retriever_chunks.retrieve(q, top_k=top_k)
        if ranked and ranked[0].doc_id == chunk_id:
            top1_hits += 1
        if any(r.doc_id == chunk_id for r in ranked):
            topk_hits += 1
    n = len(triples)
    return top1_hits / n, topk_hits / n

for k in (1, 3, 5):
    top1, topk = eval_chunks_level(top_k=k)
    print(f"k={k}: Top-1={top1:.2%}; Top-{k} hit={topk:.2%}")

# ===== СРАВНЕНИЕ =====
print("\n" + "=" * 70)
print("ИТОГОВОЕ СРАВНЕНИЕ")
print("=" * 70)
ch_top1, _ = eval_chapters(top_k=1)
ch_top3, _ = eval_chapters(top_k=3)
ch_top5, _ = eval_chapters(top_k=5)
ck_top1, _ = eval_chunks_level(top_k=1)
ck_top3, _ = eval_chunks_level(top_k=3)
ck_top5, _ = eval_chunks_level(top_k=5)

print(f"Chapters: Top-1={ch_top1:.0%}, Top-3={ch_top3:.0%}, Top-5={ch_top5:.0%}")
print(f"Chunks:   Top-1={ck_top1:.0%}, Top-3={ck_top3:.0%}, Top-5={ck_top5:.0%}")

УРОВЕНЬ 1: CHAPTERS (целые файлы)
k=1: Top-1=77.50%; Top-1 hit=77.50%
k=3: Top-1=77.50%; Top-3 hit=87.50%
k=5: Top-1=77.50%; Top-5 hit=92.50%

УРОВЕНЬ 2: CHUNKS (отрывки из текстов)
k=1: Top-1=66.67%; Top-1 hit=66.67%
k=3: Top-1=66.67%; Top-3 hit=91.67%
k=5: Top-1=66.67%; Top-5 hit=91.67%

ИТОГОВОЕ СРАВНЕНИЕ
Chapters: Top-1=78%, Top-3=78%, Top-5=78%
Chunks:   Top-1=67%, Top-3=67%, Top-5=67%
